In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nht123456789/qikt-f0-s42-pykt-scripts/DKT_A_f0_preds.parquet
/kaggle/input/datasets/nht123456789/qikt-f0-s42-pykt-scripts/fix_pykt_raw.py
/kaggle/input/datasets/nht123456789/qikt-f0-s42-pykt-scripts/metrics.py
/kaggle/input/datasets/nht123456789/qikt-f0-s42-pykt-scripts/predict_quelevel.py
/kaggle/input/datasets/nht123456789/qikt-f0-s42-pykt-scripts/predict_sharded.py
/kaggle/input/datasets/nht123456789/qikt-f0-s42-pykt-scripts/pykt_to_registry.py
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_quelevel.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_question_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_question_window_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_sequences.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_sequences_quelevel.csv
/kaggle/input/datasets/nhatanhadr/xes3g5m-extended/test_window_sequences.csv
/kagg

In [2]:
%%bash
# Một ô Kaggle cho các lần chạy pyKT (bản 26/10): huấn luyện -> predict -> registry.
# dkt, akt, simplekt...: predict_sharded.py (mức KC); qikt, iekt: predict_quelevel.py.
# Mỗi fold một thư mục run, có thêm model.ckpt và 3 log (train, predict, RAM) cho E1/E2.
set -ex
cd /kaggle/working
[ -d pykt-toolkit ] || git clone -q https://github.com/pykt-team/pykt-toolkit.git
cd pykt-toolkit && git checkout -q eec087e   # ghim commit

# --- SỬA LỖI DEVICE CPU/GPU CHO TOÀN BỘ THƯ VIỆN (TRIỆT ĐỂ) ---
sed -i '/def predict_one_step(/a\        data = {k: (v.to(next(self.parameters()).device) if isinstance(v, torch.Tensor) else v) for k, v in data.items()}' pykt/models/qikt.py
sed -i 's/concept_avg = (concept_emb_sum \/ concept_num)/concept_num = concept_num.to(concept_emb_sum.device)\n        concept_avg = (concept_emb_sum \/ concept_num)/g' pykt/models/que_base_model.py
# --------------------------------------------------------------

pip install -q -e . && pip install -q einops entmax
X=/kaggle/input/datasets/nhatanhadr/xes3g5m-extended    # một Dataset cho mọi notebook
T=/kaggle/input/datasets/nht123456789/qikt-f0-s42-pykt-scripts       # script 25/10 và 26/10
REF=$T/DKT_A_f0_preds.parquet
mkdir -p data/peiyou
for f in train_valid_sequences test_question_window_sequences \
         train_valid_sequences_quelevel test_window_sequences_quelevel; do
  ln -sf $X/$f.csv data/peiyou/$f.csv             # liên kết, không chép: tiết kiệm đĩa
done
cp $T/*.py examples/ && cd examples && mkdir -p logs
M=qikt; S=42; FOLDS="0"                    # SỬA: mô hình, seed, các fold của phiên này
EXTRA=""; PRED=predict_sharded.py; PARGS=""
case $M in
  qikt) EXTRA="--output_mode=an_irt"; PRED=predict_quelevel.py; PARGS="--seed=$S" ;;
  iekt) PRED=predict_quelevel.py; PARGS="--seed=$S" ;;
esac
trap 'kill $RAM 2>/dev/null || true' EXIT       # dừng vòng ghi RAM cả khi lỗi
for F in $FOLDS; do
  L=logs/${M}_f${F}_s${S}
  ( while true; do echo "$(date +%T) $(free -g | awk '/Mem/ {print $3"/"$2}') GB"
      sleep 15; done ) > ${L}_ram.log &
  RAM=$!
  python -u wandb_${M}_train.py --dataset_name=peiyou --fold=$F --seed=$S $EXTRA \
    --use_wandb=0 --add_uuid=0 --save_dir=sm_${M}_f${F} > ${L}_train.log 2>&1
  CKPT=$(dirname $(find sm_${M}_f${F} -name config.json | head -n 1))
  
  # --- ĐO THỜI GIAN PREDICT ---
  T0=$SECONDS
  python -u $PRED --save_dir=$CKPT --n_shards=8 --bz=64 $PARGS > ${L}_predict.log 2>&1
  echo "predict_s=$((SECONDS-T0))" >> ${L}_predict.log
  # -----------------------------
  
  kill $RAM
  python pykt_to_registry.py --save_dir=$CKPT --train_log=${L}_train.log \
    --test_csv=$X/test.csv --pykt_commit=eec087e --runs_root=/kaggle/working/runs \
    --ref_preds=$REF --logs ${L}_predict.log ${L}_ram.log
  tail -3 ${L}_predict.log
  echo "RAM cao nhất fold $F: $(awk '{print $2}' ${L}_ram.log | sort -n | tail -1) GB"
done
cat /kaggle/working/runs/registry.csv

QIKT_pyKT_f0_s42_20261101-0718 {'auc': 0.8205, 'acc': 0.8296, 'rmse': 0.3495, 'brier': 0.1221, 'ece': 0.0038, 'auprc_wrong': 0.5754, 'f1_wrong': 0.4392}
window_testauc = 0.820492
window_testacc = 0.829634
predict_s=1121
RAM cao nhất fold 0: 4/31 GB
run_id,model,variant,fold,seed,auc,acc,rmse,brier,ece,auprc_wrong,f1_wrong,n_epochs,sec_per_epoch,train_time_s,peak_mem_gb,n_params,preds_path,preds_raw_path
QIKT_pyKT_f0_s42_20261101-0718,QIKT,pyKT,0,42,0.820492,0.829634,0.349472,0.122131,0.003835,0.575356,0.439243,12,164.16666666666666,1970.0,,13290684,QIKT_pyKT_f0_s42_20261101-0718/preds.parquet,


+ cd /kaggle/working
+ '[' -d pykt-toolkit ']'
+ git clone -q https://github.com/pykt-team/pykt-toolkit.git
+ cd pykt-toolkit
+ git checkout -q eec087e
+ sed -i '/def predict_one_step(/a\        data = {k: (v.to(next(self.parameters()).device) if isinstance(v, torch.Tensor) else v) for k, v in data.items()}' pykt/models/qikt.py
+ sed -i 's/concept_avg = (concept_emb_sum \/ concept_num)/concept_num = concept_num.to(concept_emb_sum.device)\n        concept_avg = (concept_emb_sum \/ concept_num)/g' pykt/models/que_base_model.py
+ pip install -q -e .
+ pip install -q einops entmax
+ X=/kaggle/input/datasets/nhatanhadr/xes3g5m-extended
+ T=/kaggle/input/datasets/nht123456789/qikt-f0-s42-pykt-scripts
+ REF=/kaggle/input/datasets/nht123456789/qikt-f0-s42-pykt-scripts/DKT_A_f0_preds.parquet
+ mkdir -p data/peiyou
+ for f in train_valid_sequences test_question_window_sequences train_valid_sequences_quelevel test_window_sequences_quelevel
+ ln -sf /kaggle/input/datasets/nhatanhadr/xes3g5m-extend